# E1.1 · Why point-in-time control testing fails for AI

**Function E — Governance, Risk, Compliance & the CISO Office → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

---

**Risk.** An annual review certifies nothing about a system that changed on Tuesday.

**Control.** Continuous assurance; control effectiveness redefined for probabilistic systems.

**This lab.** Watch control evidence go stale without a code change.

| | |
|---|---|
| Open-source tooling | promptfoo |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E1.1"))

Point-in-time control testing fails for AI because the thing you tested is not the thing running next week — and none of the changes that break it are code changes.

In [ ]:
from cybercommons import grc
import time

now = time.time()
tests = [
    grc.ControlTest("AC-1", True, "delegation trace captured", tested_at=now - 3 * 86400),
    grc.ControlTest("SB-1", True, "egress denial log",         tested_at=now - 40 * 86400),
    grc.ControlTest("EV-1", True, "audit sample",              tested_at=now - 200 * 86400),
    grc.ControlTest("DR-1", False, "no drift alerting deployed"),
]
required = ["AC-1", "AC-2", "SB-1", "EV-1", "DR-1", "ST-1"]
v = grc.verify_continuously(tests, required, now=now)

print(f"{'control':10s}{'state':14s}age (days)")
for r in v["rows"]:
    print(f"{r['control']:10s}{r['state']:14s}{r['age_days']}")
print(f"\ncurrently evidenced {v['currently_evidenced']}/{v['required']} "
      f"= {v['coverage']:.0%}")
print(v["note"])

A point-in-time report would count AC-1, SB-1 and EV-1 as passes and claim 50%. The honest number is 17%, because two of those tests are older than their freshness window and two controls have no evidence at all.

### Expect

AC-1 is PASS, SB-1 and EV-1 are STALE, DR-1 is FAIL, AC-2 and ST-1 have NO EVIDENCE — coverage 0.167.

### Your turn

Set a freshness window per control from how fast the thing it tests actually changes. Egress policy drifts slowly; a tool manifest drifts weekly.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E1.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*